# 06 — Mechanism-threshold calibration
Labeled set: monitoring-relevant (IVN + adjacent) vs irrelevant. Run on server. Output = score table to pick `mechanism_edge`.

In [ ]:
from sentence_transformers import SentenceTransformer
import yaml
m = SentenceTransformer('zeroentropy/zembed-1-embedding', trust_remote_code=True)
lens = yaml.safe_load(open('config/lenses/monitoring.yaml'))
probes = [p for v in lens['mechanisms'].values() for p in (v.get('probes') or v if isinstance(v, list) else [])]
print(len(probes), 'probe phrases')

In [ ]:
THESIS = open('config/scopes/ivn.yaml').read().split('thesis_statements:')[1].split('\n')[1]
POS = [
 ('ivn-kd-open', 'Knowledge distillation for open-set CAN intrusion detection with energy-based out-of-distribution scoring on ECUs'),
 ('ivn-drift', 'Detecting concept drift in CAN traffic under firmware updates with adaptive threshold recalibration'),
 ('iot-abstain', 'Uncertainty-based abstention for IoT intrusion detection: selective classification with review queue'),
 ('nids-calib', 'Calibrated confidence and conformal prediction for network intrusion alerts with false-alarm guarantees'),
 ('edge-drift', 'Online domain adaptation for TinyML anomaly detectors under sensor drift'),
 ('nids-shift', 'Cross-dataset evaluation of NIDS under traffic shift with few-shot recalibration'),
]
NEG = [
 ('neg-hair', 'HairCLIP text-guided hair editing with StyleGAN modulation'),
 ('neg-protein', 'Protein structure prediction with attention networks for drug discovery'),
 ('neg-legal', 'Summarization of legal contracts with citation grounding'),
 ('neg-accuracy', 'A convolutional classifier reaching 99.9% accuracy on CAN attack classification (no monitoring mechanism)'),
 ('neg-survey', 'A survey of automotive cybersecurity standards and regulations'),
]
print(len(POS), 'pos,', len(NEG), 'neg')

In [ ]:
import numpy as np
P = m.encode(['query: ' + p for p in probes], normalize_embeddings=True)
D = m.encode(['passage: ' + t for _, t in POS + NEG], normalize_embeddings=True)
T = m.encode(['query: ' + THESIS], normalize_embeddings=True)
mech = (D @ P.T).max(axis=1)
thes = (D @ T.T).ravel()
print(f"{'id':<14}{'thesis':>8}{'mech':>8}  label")
for (pid, _), th, me in zip(POS + NEG, thes, mech):
    lab = 'POS' if pid in dict(POS) else 'NEG'
    print(f'{pid:<14}{th:>8.3f}{me:>8.3f}  {lab}')
print('\npropose mechanism_edge between max-NEG-mech and min-POS-mech above')